In [11]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np

from dotenv import load_dotenv
import os

from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [12]:
# ============================================
# LOAD DATASETS
# ============================================

providers = pd.read_csv("Data/providers_data.csv")
receivers = pd.read_csv("Data/receivers_data.csv")
food_listings = pd.read_csv("Data/food_listings_data.csv")
claims = pd.read_csv("Data/claims_data.csv")

print("All datasets loaded successfully!\n")

print("Providers:", providers.shape)
print("Receivers:", receivers.shape)
print("Food Listings:", food_listings.shape)
print("Claims:", claims.shape)

All datasets loaded successfully!

Providers: (1000, 6)
Receivers: (1000, 5)
Food Listings: (1000, 9)
Claims: (1000, 5)


In [13]:
# ============================================
# DATA TYPE CONVERSIONS
# ============================================

# Food Listings — convert Expiry_Date to datetime
food_listings["Expiry_Date"] = pd.to_datetime(
    food_listings["Expiry_Date"],
    format="%m/%d/%Y"
)

# Claims — convert Timestamp to datetime
claims["Timestamp"] = pd.to_datetime(
    claims["Timestamp"],
    format="%m/%d/%Y %H:%M"
)

# Verify conversions
print("food_listings dtypes after conversion:")
print(food_listings[["Food_ID", "Expiry_Date"]].dtypes)
print()
print("claims dtypes after conversion:")
print(claims[["Claim_ID", "Timestamp"]].dtypes)

food_listings dtypes after conversion:
Food_ID                 int64
Expiry_Date    datetime64[us]
dtype: object

claims dtypes after conversion:
Claim_ID              int64
Timestamp    datetime64[us]
dtype: object


In [14]:
# ============================================
# DATABASE CONNECTION
# ============================================

from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text

# Load .env
load_dotenv()

# Read credentials
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Connection string
connection_string = (
    f"postgresql+psycopg2://"
    f"{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Engine
engine = create_engine(connection_string)

# Test
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Connection successful:", result.fetchone())

Connection successful: (1,)


In [15]:
# ============================================
# LOAD CLEANED DATA INTO POSTGRESQL
# ============================================

tables = {
    "providers": providers,
    "receivers": receivers,
    "food_listings": food_listings,
    "claims": claims
}

for table_name, df in tables.items():

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False
    )

    print(f"Loaded: {table_name} — {len(df)} rows")

print("\nAll tables loaded successfully.")

Loaded: providers — 1000 rows
Loaded: receivers — 1000 rows
Loaded: food_listings — 1000 rows
Loaded: claims — 1000 rows

All tables loaded successfully.


In [16]:
# ============================================
# VERIFY TABLES & ROW COUNTS
# ============================================

with engine.connect() as conn:

    tables = conn.execute(
        text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
        """)
    )

    print("Tables in PostgreSQL:\n")

    for table in tables:
        print(table[0])

print("\nRow Counts\n")

with engine.connect() as conn:

    for table in [
        "providers",
        "receivers",
        "food_listings",
        "claims"
    ]:

        result = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        )

        print(f"{table}: {result.scalar()}")

Tables in PostgreSQL:

claims
food_listings
providers
receivers

Row Counts

providers: 1000
receivers: 1000
food_listings: 1000
claims: 1000


In [17]:
# ============================================
# FEATURE ENGINEERING — ANALYSIS DATASET
# ============================================

# Merge claims with food_listings to get Expiry_Date alongside each claim
claims_analysis = claims.merge(
    food_listings[["Food_ID", "Expiry_Date", "Food_Type", "Meal_Type"]],
    on="Food_ID",
    how="left"
)

# Calculate Days_Until_Expiry (positive = claimed before expiry, negative = after)
claims_analysis["Days_Until_Expiry"] = (
    claims_analysis["Expiry_Date"] - claims_analysis["Timestamp"]
).dt.days

# Summary
print("claims_analysis shape:", claims_analysis.shape)
print()
print("Days_Until_Expiry distribution:")
print(claims_analysis["Days_Until_Expiry"].describe().round(2))
print()
print("Claims BEFORE expiry:", (claims_analysis["Days_Until_Expiry"] > 0).sum())
print("Claims ON expiry day:", (claims_analysis["Days_Until_Expiry"] == 0).sum())
print("Claims AFTER expiry :", (claims_analysis["Days_Until_Expiry"] < 0).sum())

claims_analysis shape: (1000, 9)

Days_Until_Expiry distribution:
count    1000.00
mean       10.89
std         7.14
min        -6.00
25%         6.00
50%        11.00
75%        16.00
max        28.00
Name: Days_Until_Expiry, dtype: float64

Claims BEFORE expiry: 925
Claims ON expiry day: 20
Claims AFTER expiry : 55


# Data Cleaning Summary

## Datasets Loaded
| Dataset | Rows | Columns | Nulls | Duplicate IDs |
|---|---|---|---|---|
| Providers | 1000 | 6 | 0 | 0 |
| Receivers | 1000 | 5 | 0 | 0 |
| Food Listings | 1000 | 9 | 0 | 0 |
| Claims | 1000 | 5 | 0 | 0 |

## Transformations Applied
- `Expiry_Date` converted from string to datetime (format: `%m/%d/%Y`)
- `Timestamp` converted from string to datetime (format: `%m/%d/%Y %H:%M`)
- No missing values or duplicate primary keys detected across all datasets

## Feature Engineering
- Created `claims_analysis` by merging `claims` and `food_listings` on `Food_ID`
- Engineered `Days_Until_Expiry` to measure gap between claim timestamp and food expiry date

## Key Findings
- City fields contain synthetic data (~96% unique values) — city-level insights should be interpreted cautiously
- Duplicate provider and receiver names represent distinct entities with unique IDs — not a data quality issue
- `Food_Name`, `Food_Type`, and `Meal_Type` are independent categorical dimensions, not hierarchical
- 92.5% of claims (925) occurred before food expiry — consistent with intended platform behaviour
- 5.5% of claims (55) occurred after expiry, including 15 completed claims — warrants attention in a real-world system

## PostgreSQL Load
- All 4 tables loaded into `food_wastage_db` via SQLAlchemy
- `Expiry_Date` and `Timestamp` stored as `timestamp without time zone`
- Identifier columns stored as `bigint`